### Implementação dos Modelos de Aprendizado de Máquina (Random Forest, XGBoost, KNN e SVR)

#### Depois do baseline com Regressão Linear Múltipla (05), a ideia aqui é testar modelos mais flexíveis e ver se eles conseguem bater a régua que definimos: **MAE de 5,4 dias e R² de 0,41** no teste.

Para a comparação ser justa, todos os modelos usam **exatamente a mesma base, as mesmas variáveis e a mesma divisão de treino e teste** do baseline (treino até março/2018, teste de abril a agosto/2018). A única coisa que muda é o algoritmo.

In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score

def root_mean_squared_error(y_real, y_prev):
    return np.sqrt(mean_squared_error(y_real, y_prev))
from xgboost import XGBRegressor  # pip install xgboost

-----------------------------------------------------------------------------------------------------

#### I. Recriando o mesmo X e y do baseline

Carrego o `df_modelo.csv` que o 05 salvou (já com uma linha por pedido) e repito os mesmos passos de lá:
* mesmo corte de data;
* `log1p` nas variáveis com cauda longa;
* estados virando dummies;
* sem o `percentual_frete` (que tiramos por causa do VIF);
* alvo do treino limitado entre os percentis 1% e 99%, com o teste intacto.

Deixei tudo numa função só, para não ter risco de algum modelo receber os dados de um jeito diferente.

In [3]:
pedidos = pd.read_csv("../data/processed/df_modelo.csv")
pedidos["order_purchase_timestamp"] = pd.to_datetime(pedidos["order_purchase_timestamp"])

alvo = "dias_atrasados"
numericas = [
    "janela_prometida", "distancia_km", "cross_state", "taxa_atraso_vendedor",
    "volume_historico_vendedor", "preco_total", "frete_total", "peso_total_g",
    "volume_total_cm3", "quantidade_itens_pedido", "vendedores_distintos",
    "payment_installments", "dia_semana_compra", "epoca_alta_demanda",
]
dummies = [c for c in pedidos.columns
           if c.startswith("product_category_name_english_") or c.startswith("payment_type_")]
estados = ["customer_state", "seller_state"]
assimetricas = ["preco_total", "frete_total", "peso_total_g", "volume_total_cm3",
                "distancia_km", "volume_historico_vendedor"]

corte = "2018-04-01"
treino = pedidos[pedidos["order_purchase_timestamp"] < corte].copy()
teste = pedidos[pedidos["order_purchase_timestamp"] >= corte].copy()

def monta_X(base, medias):
    X = base[numericas + dummies + estados].copy()
    for col, media in medias.items():
        X[col] = X[col].fillna(media)
    for col in assimetricas:
        X[col] = np.log1p(X[col])
    return X

medias_treino = treino[["distancia_km", "taxa_atraso_vendedor"]].mean().to_dict()
X_treino = pd.get_dummies(monta_X(treino, medias_treino), columns=estados, drop_first=True, dtype=float)
X_teste = pd.get_dummies(monta_X(teste, medias_treino), columns=estados, drop_first=True, dtype=float)
X_teste = X_teste.reindex(columns=X_treino.columns, fill_value=0)
X_treino, X_teste = X_treino.astype(float), X_teste.astype(float)

limite_inf, limite_sup = treino[alvo].quantile([0.01, 0.99])
y_treino = treino[alvo].clip(limite_inf, limite_sup)
y_teste = teste[alvo]

print(f"Treino: {X_treino.shape} | Teste: {X_teste.shape}")

Treino: (64303, 135) | Teste: (32150, 135)


KNN e SVR trabalham com distâncias, então precisam das variáveis na mesma escala (senão o preço, que vai até milhares, "manda" sozinho). Random Forest e XGBoost são baseados em árvores e não ligam para escala, então uso os dados sem padronizar neles. O scaler é ajustado só no treino.

In [4]:
scaler = StandardScaler()
X_treino_esc = scaler.fit_transform(X_treino)
X_teste_esc = scaler.transform(X_teste)

-----------------------------------------------------------------------------------------------------

#### II. Uma função para treinar e avaliar todo mundo do mesmo jeito

Além de MAE, RMSE e R² (as mesmas do baseline), guardo o tempo de treino e a ROC-AUC da pergunta "vai atrasar?", usando a previsão como nível de risco. Para começar, reproduzo a regressão linear aqui, só para confirmar que os números batem com o 05.

In [5]:
resultados = {}
previsoes = {}
modelos = {}

def avalia(nome, modelo, X_tr, X_te, y_tr=y_treino):
    inicio = time.time()
    modelo.fit(X_tr, y_tr)
    tempo = time.time() - inicio
    prev = modelo.predict(X_te)
    resultados[nome] = {
        "MAE (dias)": mean_absolute_error(y_teste, prev),
        "RMSE (dias)": root_mean_squared_error(y_teste, prev),
        "R²": r2_score(y_teste, prev),
        "ROC-AUC atraso": roc_auc_score((y_teste > 0).astype(int), prev),
        "Tempo de treino (s)": tempo,
    }
    previsoes[nome] = prev
    modelos[nome] = modelo
    return pd.Series(resultados[nome]).round(3)

avalia("Regressão Linear (baseline)", LinearRegression(), X_treino_esc, X_teste_esc)

MAE (dias)             5.407
RMSE (dias)            7.184
R²                     0.406
ROC-AUC atraso         0.754
Tempo de treino (s)    0.949
dtype: float64

Bateu com o 05 (MAE 5,41 e R² 0,41). Agora sim, os modelos novos.

-----------------------------------------------------------------------------------------------------

#### III. Random Forest

A Random Forest treina várias árvores de decisão, cada uma com uma amostra diferente dos dados e das variáveis, e tira a média das previsões. Ela consegue capturar relações não lineares e interações (por exemplo, "distância grande **e** época de Black Friday") que a regressão linear não pega.

Hiperparâmetros que escolhi (sem tuning ainda, é um experimento preliminar):
* `n_estimators=200`: 200 árvores, um meio-termo entre qualidade e tempo.
* `min_samples_leaf=20`: cada folha precisa de pelo menos 20 pedidos, o que evita que a árvore decore casos isolados.
* `max_features=0.5`: cada divisão olha metade das variáveis, para as árvores ficarem mais diferentes entre si.

In [6]:
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=20, max_features=0.5,
                           n_jobs=-1, random_state=42)
avalia("Random Forest", rf, X_treino, X_teste)

MAE (dias)              5.689
RMSE (dias)             7.495
R²                      0.353
ROC-AUC atraso          0.739
Tempo de treino (s)    67.885
dtype: float64

-----------------------------------------------------------------------------------------------------

#### IV. KNN

O KNN é o mais intuitivo: para prever o atraso de um pedido, ele procura os `k` pedidos mais parecidos no treino e tira a média do atraso deles.

O ponto chato é escolher o `k`. Não quis escolher olhando o teste, porque isso seria "colar" na prova. Então separei os últimos 3 meses do treino (jan–mar/2018) como **validação**, testei alguns valores de `k` ali e só depois treinei com o treino inteiro.

In [7]:
datas_treino = treino["order_purchase_timestamp"].values
eh_validacao = datas_treino >= np.datetime64("2018-01-01")

X_sub, X_val = X_treino_esc[~eh_validacao], X_treino_esc[eh_validacao]
y_sub, y_val = y_treino[~eh_validacao], treino[alvo][eh_validacao]

busca_k = {}
for k in [5, 15, 25, 50, 100]:
    knn_teste = KNeighborsRegressor(n_neighbors=k, weights="distance", n_jobs=-1).fit(X_sub, y_sub)
    busca_k[k] = mean_absolute_error(y_val, knn_teste.predict(X_val))

busca_k = pd.Series(busca_k, name="MAE na validação")
melhor_k = busca_k.idxmin()
print(f"Melhor k na validação: {melhor_k}")
busca_k.round(3).to_frame()

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] O sistema não pode encontrar o arquivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_f

Melhor k na validação: 50


,MAE na validação
5,7.480
15,7.212
25,7.185
50,7.147
100,7.151


In [8]:
knn = KNeighborsRegressor(n_neighbors=melhor_k, weights="distance", n_jobs=-1)
avalia(f"KNN (k={melhor_k})", knn, X_treino_esc, X_teste_esc)

MAE (dias)             6.694
RMSE (dias)            9.087
R²                     0.049
ROC-AUC atraso         0.628
Tempo de treino (s)    0.117
dtype: float64

Desconfio que as ~120 colunas de dummies (categoria e estado) atrapalham o KNN. Com tantas dimensões, "estar perto" vira quase só "ser da mesma categoria e do mesmo estado". Como teste rápido, rodo o KNN só com as variáveis numéricas:

In [9]:
idx_num = [X_treino.columns.get_loc(c) for c in numericas]
knn_num = KNeighborsRegressor(n_neighbors=melhor_k, weights="distance", n_jobs=-1)
avalia(f"KNN (k={melhor_k}, só numéricas)", knn_num, X_treino_esc[:, idx_num], X_teste_esc[:, idx_num])

MAE (dias)             5.954
RMSE (dias)            7.911
R²                     0.280
ROC-AUC atraso         0.700
Tempo de treino (s)    0.362
dtype: float64

-----------------------------------------------------------------------------------------------------

#### VI. SVR

O SVR (Support Vector Regression) tenta achar uma função que fique dentro de uma "margem" de erro ε para a maioria dos pontos, ignorando erros pequenos. Com o kernel RBF, ele também captura relações não lineares.

O problema prático: o tempo de treino do SVR cresce mais ou menos com o **quadrado** do número de linhas, e com 64 mil pedidos ele demoraria muito. Por isso treino com uma **amostra aleatória de 10 mil pedidos** do treino, o que já é bastante para o SVR. O teste continua sendo o teste completo.

* `C=1`: o quanto o modelo é penalizado por erros fora da margem.
* `epsilon=1`: erros de até 1 dia são tolerados.

In [10]:
amostra_svr = np.random.default_rng(42).choice(len(X_treino_esc), 10_000, replace=False)

svr = SVR(kernel="rbf", C=1.0, epsilon=1.0)
avalia("SVR (amostra de 10 mil)", svr, X_treino_esc[amostra_svr], X_teste_esc,
       y_tr=y_treino.iloc[amostra_svr])

MAE (dias)              5.086
RMSE (dias)             7.216
R²                      0.401
ROC-AUC atraso          0.691
Tempo de treino (s)    13.975
dtype: float64

-----------------------------------------------------------------------------------------------------

#### VII. Registro dos resultados

Tabela com as métricas de todos os modelos no conjunto de teste. A análise desses resultados e a comparação detalhada com o baseline ficam para a próxima etapa do projeto (Semana 8).

In [11]:
tabela = pd.DataFrame(resultados).T.sort_values("MAE (dias)")
tabela.round(3)

,MAE (dias),RMSE (dias),R²,ROC-AUC atraso,Tempo de treino (s)
SVR (amostra de 10 mil),5.086,7.216,0.401,0.691,13.975
Regressão Linear (baseline),5.407,7.184,0.406,0.754,0.949
Random Forest,5.689,7.495,0.353,0.739,67.885
"KNN (k=50, só numéricas)",5.954,7.911,0.280,0.700,0.362
KNN (k=50),6.694,9.087,0.049,0.628,0.117


In [12]:
pd.DataFrame(resultados).T.round(4).to_csv("../data/processed/resultados_modelos_ml.csv")
print("Salvo: resultados_modelos_ml.csv")

Salvo: resultados_modelos_ml.csv


-----------------------------------------------------------------------------------------------------

#### Próximos passos

Com os quatro modelos implementados e a primeira rodada de experimentos registrada, o próximo passo é analisar esses resultados: verificar em que tipo de pedido cada modelo erra mais e comparar o MAE e o RMSE de cada um com os da Regressão Linear (baseline).